In [1]:
# import libraries
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

#load dataset
data = load_breast_cancer()

X = data.data
y = data.target

print(X.shape)
print(y.shape)

#Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Feature Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

(569, 30)
(569,)


In [2]:
#QDA
class QDA:

    def fit(self, X, y):

        self.classes = np.unique(y)

        self.means = {}
        self.priors = {}
        self.covariances = {}
        self.inv_covariances = {}
        self.det_covariances = {}

        for c in self.classes:

            X_c = X[y == c]

            self.means[c] = np.mean(X_c, axis=0)

            self.priors[c] = len(X_c) / len(X)

            cov = np.cov(X_c, rowvar=False)

            # Small regularization
            cov += 1e-6 * np.eye(cov.shape[0])

            self.covariances[c] = cov

            self.inv_covariances[c] = np.linalg.inv(cov)

            self.det_covariances[c] = np.linalg.det(cov)

    def _discriminant(self, x, c):

        mean = self.means[c]

        inv_cov = self.inv_covariances[c]

        det_cov = self.det_covariances[c]

        prior = self.priors[c]

        diff = x - mean

        term1 = -0.5 * np.log(det_cov)

        term2 = -0.5 * (diff @ inv_cov @ diff)

        term3 = np.log(prior)

        return term1 + term2 + term3

    def predict(self, X):

        predictions = []

        for x in X:

            scores = [
                self._discriminant(x, c)
                for c in self.classes
            ]

            predictions.append(
                self.classes[np.argmax(scores)]
            )

        return np.array(predictions)

In [3]:
#Train
model = QDA()
model.fit(X_train, y_train)

In [4]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy: 0.956140350877193
[[41  2]
 [ 3 68]]
